# 04 — Graph RAG

```
╔═══════════════════════════════════════════════════════════════════════╗
║                       4. GRAPH RAG                                    ║
║                                                                       ║
║  Documents ──► Chunks ──► Embedding Model ──► Vector Database        ║
║                    │                                  │               ║
║                    │                             top-k│               ║
║                    ▼                                  ▼               ║
║            LLM Graph Generator ──────────────► Graph Database        ║
║            (extract entities                 (Neo4j: nodes+edges     ║
║             & relations)                      + vector index)         ║
║                                                       │               ║
║  Query ────────────────────────────────► seed nodes  │               ║
║                                          + traverse  │               ║
║                                                       ▼               ║
║  Response ◄── Generative Model ◄── Prompt ◄── Context (Graph+Vectors)║
╚═══════════════════════════════════════════════════════════════════════╝
```

## The problem Graph RAG solves

Naive RAG retrieves chunks independently. But many real questions require **connecting facts across multiple documents** — a multi-hop path:

> "Who should I contact about the permanent fix for the Joint 4 overheating problem?"

This requires:
1. `spec_arm_v2.txt` → Joint 4 issue → see `INC-2024-031`
2. `incident_inc2024031.txt` → permanent fix is in PROJ-TITAN
3. `project_titan.txt` → PROJ-TITAN is owned by Dr. Amara Nwosu and led by Lena Bergström
4. `team_engineering.txt` → Lena Bergström's email is lena.bergstrom@helios-robotics.com

A knowledge graph makes these **relationships explicit** so we can traverse them.

## Architecture
- **LLM entity extraction**: parse documents, extract entities (people, products, incidents, projects) and their relationships
- **Neo4j**: store the graph (nodes + edges) AND node embeddings in a single store (Neo4j 5 native vector index)
- **Retrieval**: embed query → find seed nodes via vector similarity → traverse graph (Cypher) for multi-hop context
- **Fallback**: `networkx` in-process graph if Neo4j/Docker isn't running

## Setup

In [ ]:
import sys; sys.path.insert(0, '..')
import ragkit.config as cfg

cfg.BACKEND = "claude"   # "claude" | "local"

# ── Neo4j toggle ─────────────────────────────────────────────────────────────
# Set USE_NEO4J = False to use the networkx fallback (no Docker needed)
USE_NEO4J = True

print(f"Backend: {cfg.BACKEND}  |  Device: {cfg.DEVICE}")
print(f"Graph backend: {'Neo4j (Docker)' if USE_NEO4J else 'networkx (in-process)'}")

In [ ]:
# ── Check Neo4j connectivity ─────────────────────────────────────────────────
if USE_NEO4J:
    from neo4j import GraphDatabase
    try:
        driver = GraphDatabase.driver(
            cfg.NEO4J_URI, auth=(cfg.NEO4J_USER, cfg.NEO4J_PASSWORD)
        )
        driver.verify_connectivity()
        print(f"Neo4j: ✅  Connected to {cfg.NEO4J_URI}")
        print("Neo4j Browser: http://localhost:7474  (user: neo4j, pass: helios-rag-2025)")
    except Exception as e:
        print(f"Neo4j: ❌  {e}")
        print()
        print("To start Neo4j with Docker:")
        print("  docker run -d --name helios-neo4j \\")
        print("    -p 7474:7474 -p 7687:7687 \\")
        print("    -e NEO4J_AUTH=neo4j/helios-rag-2025 \\")
        print("    -e NEO4J_PLUGINS='[\"apoc\"]' \\")
        print("    neo4j:5")
        print()
        print("Falling back to networkx...")
        USE_NEO4J = False

## Step 1 — Extract entities and relationships with an LLM

We use the LLM to parse each document and extract a structured graph: nodes (entities) and edges (relationships).

In [ ]:
import json
from ragkit.llm import generate
from ragkit.data import load_corpus

EXTRACT_SYSTEM = """You are a knowledge graph extraction assistant. 
Extract entities and relationships from the provided text.
Return ONLY valid JSON, nothing else."""

EXTRACT_TEMPLATE = """Extract entities and relationships from this text.

Entity types to look for:
- Product: physical products (arms, bases, controllers, parts)
- Person: people (engineers, managers)
- Incident: incident reports (INC-YYYY-NNN)
- Project: projects (PROJ-XXX)
- Firmware: firmware versions (FW-XXX)
- Team: teams or departments

Relationship types:
- CAUSES, FIXES, AFFECTS, OWNS, WORKS_ON, PART_OF, RELATED_TO, ASSIGNED_TO, REQUIRES

Text:
{text}

Return JSON with this exact structure:
{{
  "entities": [
    {{"id": "unique_id", "name": "entity name", "type": "Product|Person|Incident|Project|Firmware|Team", "properties": {{"part_number": "...", "description": "one sentence"}}}}
  ],
  "relationships": [
    {{"source": "entity_id", "target": "entity_id", "type": "RELATIONSHIP_TYPE", "description": "brief description"}}
  ]
}}"""

def extract_graph(text: str, source: str) -> dict:
    prompt = EXTRACT_TEMPLATE.format(text=text[:3000])  # limit per call
    raw = generate(prompt, system=EXTRACT_SYSTEM, max_tokens=2048)
    # Strip markdown code fences if present
    raw = raw.strip()
    if raw.startswith('```'):
        raw = raw.split('\n', 1)[1]
        if raw.endswith('```'):
            raw = raw.rsplit('```', 1)[0]
    try:
        data = json.loads(raw)
        for e in data.get('entities', []):
            e['source_doc'] = source
        return data
    except json.JSONDecodeError as err:
        print(f"  Warning: JSON parse error for {source}: {err}")
        return {'entities': [], 'relationships': []}

# Extract from a subset of key documents
key_docs = ['spec_arm_v2.txt', 'incident_inc2024031.txt', 'project_titan.txt',
            'team_engineering.txt', 'spec_firmware_release_notes.txt']

corpus = load_corpus()
docs_to_extract = [d for d in corpus if d['source'] in key_docs]

all_entities, all_relationships = {}, []

for doc in docs_to_extract:
    print(f"Extracting from {doc['source']}...")
    result = extract_graph(doc['text'], doc['source'])
    
    for e in result.get('entities', []):
        # Use name as dedup key
        key = e['name'].lower().replace(' ', '_')
        if key not in all_entities:
            all_entities[key] = e
            all_entities[key]['id'] = key
    
    all_relationships.extend(result.get('relationships', []))
    print(f"  → {len(result.get('entities',[]))} entities, {len(result.get('relationships',[]))} relationships")

print(f"\nTotal: {len(all_entities)} unique entities, {len(all_relationships)} relationships")

## Step 2a — Store in Neo4j (primary path)

In [ ]:
if USE_NEO4J:
    from ragkit.embeddings import embed
    import numpy as np
    
    entity_list = list(all_entities.values())
    entity_names = [e['name'] + " — " + e.get('properties', {}).get('description', '') for e in entity_list]
    
    print("Embedding entity descriptions...")
    entity_vecs = embed(entity_names)
    print(f"Entity embeddings: {entity_vecs.shape}")
    
    with driver.session() as session:
        # Clear existing graph
        session.run("MATCH (n) DETACH DELETE n")
        
        # Create nodes
        for e, vec in zip(entity_list, entity_vecs):
            session.run(
                """MERGE (n:Entity {id: $id})
                SET n.name = $name, n.type = $type, n.source_doc = $source,
                    n.description = $desc, n.embedding = $emb""",
                id=e['id'],
                name=e['name'],
                type=e['type'],
                source=e.get('source_doc', ''),
                desc=e.get('properties', {}).get('description', ''),
                emb=vec.tolist()
            )
        
        # Create vector index
        try:
            session.run("""
                CREATE VECTOR INDEX entity_embedding IF NOT EXISTS
                FOR (n:Entity) ON (n.embedding)
                OPTIONS {indexConfig: {
                    `vector.dimensions`: 384,
                    `vector.similarity_function`: 'cosine'
                }}
            """)
        except Exception as idx_err:
            print(f"  Note: vector index: {idx_err}")
        
        # Create relationships
        rel_created = 0
        for rel in all_relationships:
            src_key = rel['source'].lower().replace(' ', '_')
            tgt_key = rel['target'].lower().replace(' ', '_')
            if src_key in all_entities and tgt_key in all_entities:
                session.run(
                    f"""MATCH (a:Entity {{id: $src}}), (b:Entity {{id: $tgt}})
                       MERGE (a)-[r:{rel['type']} {{desc: $desc}}]->(b)""",
                    src=src_key, tgt=tgt_key, desc=rel.get('description', '')
                )
                rel_created += 1
    
    print(f"\nNeo4j: {len(entity_list)} nodes, {rel_created} relationships created")
    print("Open Neo4j Browser → http://localhost:7474 and run: MATCH (n)-[r]->(m) RETURN n,r,m LIMIT 50")

## Step 2b — Store in networkx (fallback path)

In [ ]:
import networkx as nx

# Build networkx graph regardless (for visualisation and fallback)
G = nx.DiGraph()

for e in all_entities.values():
    G.add_node(e['id'], **e)

for rel in all_relationships:
    src_key = rel['source'].lower().replace(' ', '_')
    tgt_key = rel['target'].lower().replace(' ', '_')
    if src_key in G.nodes and tgt_key in G.nodes:
        G.add_edge(src_key, tgt_key, rel_type=rel['type'], desc=rel.get('description', ''))

print(f"networkx graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

## Step 3 — Visualise the knowledge graph

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.cm as cm

type_colors = {
    'Product': '#2b5797', 'Person': '#27ae60', 'Incident': '#e74c3c',
    'Project': '#f39c12', 'Firmware': '#8e44ad', 'Team': '#16a085'
}

fig, ax = plt.subplots(figsize=(14, 9))

node_colors = [type_colors.get(G.nodes[n].get('type', 'Product'), '#aaa') for n in G.nodes]
labels = {n: G.nodes[n].get('name', n)[:18] for n in G.nodes}

pos = nx.spring_layout(G, seed=42, k=2.5)
nx.draw_networkx_nodes(G, pos, ax=ax, node_color=node_colors, node_size=800, alpha=0.9)
nx.draw_networkx_labels(G, pos, labels, ax=ax, font_size=6)
nx.draw_networkx_edges(G, pos, ax=ax, edge_color='#888', arrows=True,
                       arrowsize=15, connectionstyle='arc3,rad=0.1')
edge_labels = {(u, v): G.edges[u, v]['rel_type'] for u, v in G.edges}
nx.draw_networkx_edge_labels(G, pos, edge_labels, ax=ax, font_size=5, label_pos=0.4)

# Legend
patches = [plt.matplotlib.patches.Patch(color=c, label=t) for t, c in type_colors.items()]
ax.legend(handles=patches, loc='upper left', fontsize=8)
ax.set_title('Helios Robotics Knowledge Graph', fontsize=13)
ax.axis('off')
plt.tight_layout(); plt.show()

## Step 4 — Graph retrieval: seed + traverse

In [ ]:
from ragkit.embeddings import embed
from ragkit.vectorstore import build_collection, query_collection

# Also maintain a Chroma vector index of chunks (for the vector half of the pipeline)
from ragkit.data import build_chunked_corpus
texts, metas = build_chunked_corpus(chunk_size=200, overlap=40)
chunk_collection = build_collection("helios_graph_chunks", texts, metas, persist_dir="../.chroma")
print(f"Chunk collection: {chunk_collection.count()} chunks")

def find_seed_nodes_neo4j(query: str, top_k: int = 3) -> list[dict]:
    """Find seed nodes via Neo4j vector similarity search."""
    q_vec = embed([query])[0].tolist()
    with driver.session() as session:
        result = session.run("""
            CALL db.index.vector.queryNodes('entity_embedding', $k, $vec)
            YIELD node, score
            RETURN node.id AS id, node.name AS name, node.type AS type,
                   node.description AS desc, score
            ORDER BY score DESC
        """, k=top_k, vec=q_vec)
        return [dict(r) for r in result]

def find_seed_nodes_nx(query: str, top_k: int = 3) -> list[dict]:
    """Find seed nodes via cosine similarity over entity embeddings (networkx fallback)."""
    from ragkit.embeddings import cosine_similarity
    q_vec = embed([query])[0]
    
    node_list = list(G.nodes(data=True))
    node_names = [d.get('name', nid) + " " + d.get('properties', {}).get('description', '')
                  for nid, d in node_list]
    node_vecs = embed(node_names)
    sims = cosine_similarity(q_vec, node_vecs)
    
    top_idx = sims.argsort()[::-1][:top_k]
    seeds = []
    for i in top_idx:
        nid, data = node_list[i]
        seeds.append({'id': nid, 'name': data.get('name', nid),
                      'type': data.get('type', ''), 'score': float(sims[i])})
    return seeds

find_seed_nodes = find_seed_nodes_neo4j if USE_NEO4J else find_seed_nodes_nx

# Test seed node retrieval
multihop_q = "Who should I contact about the permanent fix for the Joint 4 overheating problem?"
seeds = find_seed_nodes(multihop_q)
print(f"Query: '{multihop_q}'")
print("Seed nodes:")
for s in seeds:
    print(f"  [{s.get('score', 0):.3f}] ({s['type']}) {s['name']}")

In [ ]:
def traverse_graph_neo4j(seed_ids: list[str], hops: int = 2) -> list[str]:
    """Traverse from seed nodes and collect connected facts (Neo4j)."""
    context_facts = []
    with driver.session() as session:
        for seed_id in seed_ids:
            result = session.run("""
                MATCH (start:Entity {id: $id})
                CALL apoc.path.subgraphNodes(start, {
                    maxLevel: $hops,
                    labelFilter: 'Entity'
                }) YIELD node
                MATCH (start)-[r*1..2]->(node)
                RETURN start.name AS from, [rel in r | type(rel)] AS rel_types,
                       node.name AS to, node.description AS desc
                LIMIT 20
            """, id=seed_id, hops=hops)
            
            for row in result:
                path_str = " → ".join(row['rel_types'])
                fact = f"{row['from']} --[{path_str}]--> {row['to']}: {row['desc']}"
                context_facts.append(fact)
    return context_facts

def traverse_graph_nx(seed_ids: list[str], hops: int = 2) -> list[str]:
    """Traverse from seed nodes and collect connected facts (networkx fallback)."""
    context_facts = []
    for seed_id in seed_ids:
        if seed_id not in G:
            continue
        # BFS up to `hops` levels
        visited = {seed_id}
        frontier = [seed_id]
        for _ in range(hops):
            next_frontier = []
            for node in frontier:
                for _, neighbor, data in G.out_edges(node, data=True):
                    if neighbor not in visited:
                        visited.add(neighbor)
                        next_frontier.append(neighbor)
                        n_data = G.nodes[neighbor]
                        fact = (f"{G.nodes[node]['name']} --[{data['rel_type']}]--> "
                                f"{n_data['name']}: "
                                f"{n_data.get('properties', {}).get('description', '')}")
                        context_facts.append(fact)
            frontier = next_frontier
    return context_facts

traverse_graph = traverse_graph_neo4j if USE_NEO4J else traverse_graph_nx

graph_context = traverse_graph([s['id'] for s in seeds], hops=2)
print(f"Graph traversal found {len(graph_context)} facts:")
for i, fact in enumerate(graph_context[:10]):
    print(f"  {i+1}. {fact[:120]}")

## Step 5 — Fuse graph + vector context and generate

In [ ]:
from ragkit.vectorstore import query_collection
from ragkit.pretty import show_graph_context, show_answer

def graph_rag(question: str, chunk_collection, hops: int = 2, vec_k: int = 5) -> str:
    # 1. Find seed nodes via graph vector index
    seeds = find_seed_nodes(question, top_k=3)
    
    # 2. Traverse graph for multi-hop context
    graph_ctx = traverse_graph([s['id'] for s in seeds], hops=hops)
    
    # 3. Vector retrieval for raw passage context
    vec_hits = query_collection(chunk_collection, question, k=vec_k)
    
    # 4. Show both context streams
    show_graph_context(vec_hits, graph_ctx)
    
    # 5. Build combined prompt
    vec_ctx_str = "\n\n".join(f"[{h.metadata['source']}]\n{h.text}" for h in vec_hits)
    graph_ctx_str = "\n".join(f"- {f}" for f in graph_ctx[:15])
    
    prompt = f"""VECTOR CONTEXT (relevant passages):
{vec_ctx_str}

GRAPH CONTEXT (entity relationships, multi-hop traversal):
{graph_ctx_str}

---
Question: {question}
Use BOTH the passage context and the relationship graph to give a complete answer."""
    
    SYSTEM = """You are a technical assistant for Helios Robotics. 
    You have access to both document passages and a knowledge graph of entity relationships.
    Use both sources to answer multi-hop questions that require connecting multiple facts."""
    
    return generate(prompt, system=SYSTEM)

# The multi-hop question that Naive RAG failed on
answer = graph_rag(multihop_q, chunk_collection)
show_answer(answer, title="Graph RAG — multi-hop question")

In [ ]:
# Compare directly with Naive RAG on the same multi-hop question
from ragkit.vectorstore import query_collection
from ragkit.llm import generate

naive_hits = query_collection(chunk_collection, multihop_q, k=5)
naive_ctx = "\n\n".join(f"[{h.metadata['source']}]\n{h.text}" for h in naive_hits)
naive_answer = generate(
    f"Context:\n{naive_ctx}\n\n---\nQuestion: {multihop_q}",
    system="You are a technical assistant. Answer only from the provided context."
)

print("=" * 60)
print("NAIVE RAG:")
print(naive_answer)
print()
print("=" * 60)
print("GRAPH RAG:")
answer2 = graph_rag(multihop_q, chunk_collection)
print(answer2)

## Exercise A — Multi-hop traversal of a drag family tree

Drag culture has **houses and lineages**: a "drag mother" mentors "drag daughters", who go on to mentor their own. Relationship questions like *"who are someone's drag grandchildren?"* are **multi-hop** — flat retrieval of a single chunk can only see direct connections. Graph RAG answers them by **traversing edges**.

(The lineage below is a simplified, illustrative example, not a record of real performers.)

1. **Predict first**: From `children` below, who are Mama Ru's direct daughters (1 hop)? Who are her grandchildren (2 hops)?
2. **Implement**: Complete `descendants(root, hops)` to walk the graph hop by hop.
3. **Reflect**: One sentence — why can't a single retrieved document answer a 2-hop "grandchildren" question?

In [ ]:
# mother -> list of daughters  (an illustrative lineage)
children = {
    "Mama Ru":        ["House Mother A", "House Mother B"],
    "House Mother A": ["Daughter A1", "Daughter A2"],
    "House Mother B": ["Daughter B1"],
}

# ── Task 1: predictions (write before running) ────────────────────────────────
#   1-hop (daughters):     ____________
#   2-hop (grandchildren): ____________

# ── Task 2: traverse exactly `hops` levels down from `root` ───────────────────
def descendants(root: str, hops: int) -> list[str]:
    frontier = [root]
    for _ in range(hops):
        frontier = [kid for node in frontier for kid in children.get(node, [])]
    return frontier

kids       = set(descendants("Mama Ru", 1))
grandkids  = set(descendants("Mama Ru", 2))
print("Daughters (1 hop):    ", kids)
print("Grandchildren (2 hops):", grandkids)

# ── Task 3: why flat retrieval fails on 2-hop questions (comment) ─────────────
#   Your answer:

# ── Self-check ────────────────────────────────────────────────────────────────
assert kids == {"House Mother A", "House Mother B"}
assert grandkids == {"Daughter A1", "Daughter A2", "Daughter B1"}
print("\n✅ Exercise A checks passed!")

## Exercise B — Finding a common ancestor in a language family tree

The same graph idea answers linguistics questions. Spanish and French are both **Romance** languages descended from **Latin**; English and German descend from **Proto-Germanic**. "What do these two languages have in common?" becomes a **shared-parent** lookup on a tree.

1. **Implement**: Build a `parent` map from the `family` tree, then complete `common_ancestor(a, b)`.
2. **Check**: Spanish + French should share Latin; Spanish + German should share nothing.
3. **Reflect**: One sentence — how is "common ancestor" the same graph operation as the drag-grandchildren question, just traversing *upward*?

In [ ]:
# parent -> children
family = {
    "Latin":          ["Spanish", "French", "Italian", "Portuguese"],
    "Proto-Germanic": ["English", "German", "Dutch"],
}

# ── Task 1: invert the tree into a child -> parent map ────────────────────────
parent = {child: p for p, kids in family.items() for child in kids}

def common_ancestor(a: str, b: str) -> str | None:
    # Return the shared parent language, or None if they do not share one.
    return parent.get(a) if parent.get(a) == parent.get(b) else None

print("Spanish & French ->", common_ancestor("Spanish", "French"))
print("Spanish & German ->", common_ancestor("Spanish", "German"))

# ── Task 3: upward vs downward traversal (comment) ────────────────────────────
#   Your answer:

# ── Self-check ────────────────────────────────────────────────────────────────
assert common_ancestor("Spanish", "French") == "Latin"
assert common_ancestor("French", "Italian") == "Latin"
assert common_ancestor("Spanish", "German") is None
print("\n✅ Exercise B checks passed!")

## Tradeoffs

| Aspect | Naive RAG | Graph RAG |
|---|---|---|
| **Multi-hop questions** | ★☆☆☆☆ | ★★★★★ |
| **Setup complexity** | ★☆☆☆☆ | ★★★★☆ (Neo4j + extraction) |
| **Extraction quality** | N/A | Depends on LLM extraction accuracy |
| **Infrastructure** | None | Neo4j (Docker) or networkx |
| **Latency** | ★★★★★ | ★★★☆☆ (graph traversal adds time) |
| **When to use** | Simple FAQs | Interconnected domains: org charts, system dependencies, research papers |

## Exercises

1. **Neo4j Browser**: Open http://localhost:7474 and run the Cypher query: `MATCH (n)-[r]->(m) RETURN n,r,m LIMIT 50`. Explore the visual graph.
2. **Add more documents**: Extract from `team_field_service.txt` and `project_fleet_ai.txt`. Do more edges appear?
3. **3-hop traversal**: Change `hops=3` in `graph_rag()`. Does the answer improve for the multi-hop question, or does it add noise?
4. **Extraction errors**: Find a case where the LLM extracted a wrong entity or relationship. How would you fix it?

**Next:** [05_hybrid_rag.ipynb](05_hybrid_rag.ipynb) — combine dense (vector) and sparse (BM25) retrieval.